# ⚙️ What is an Optimizer? Why LLMs Use AdamW

Welcome! In this notebook, we answer the fundamental question: <b>What is an optimizer, what are the options, and why does virtually every LLM use AdamW?</b>

---

## 1. The Metaphor: Navigating a Foggy Mountain at Night

Imagine standing on a steep mountain in dense fog at midnight.<br>
Your goal is to reach the <b>lowest valley (minimum loss)</b>.<br>
You can only feel the slope right beneath your feet (the gradient).

<div style='display: flex; justify-content: center; margin: 20px 0;'>
<svg width='820' height='220' viewBox='0 0 820 220' xmlns='http://www.w3.org/2000/svg' style='background: #0d1117; border: 1px solid #30363d; border-radius: 12px; font-family: -apple-system, sans-serif;'>
  <!-- Card 1: SGD -->
  <g transform='translate(20, 20)'>
    <rect width='180' height='180' rx='8' fill='#161b22' stroke='#30363d' stroke-width='1.5'/>
    <text x='15' y='28' fill='#f85149' font-size='13' font-weight='bold'>1. SGD</text>
    <text x='15' y='46' fill='#8b949e' font-size='10'>The Blind Hiker</text>
    <text x='15' y='80' fill='#c9d1d9' font-size='11'>Takes a step strictly</text>
    <text x='15' y='96' fill='#c9d1d9' font-size='11'>in slope direction.</text>
    <text x='15' y='130' fill='#f85149' font-size='10'>❌ Bounces wildly</text>
    <text x='15' y='145' fill='#f85149' font-size='10'>across steep ravines.</text>
  </g>
  <!-- Card 2: SGD + Momentum -->
  <g transform='translate(215, 20)'>
    <rect width='180' height='180' rx='8' fill='#161b22' stroke='#30363d' stroke-width='1.5'/>
    <text x='15' y='28' fill='#d29922' font-size='13' font-weight='bold'>2. Momentum</text>
    <text x='15' y='46' fill='#8b949e' font-size='10'>The Bowling Ball</text>
    <text x='15' y='80' fill='#c9d1d9' font-size='11'>Remembers past</text>
    <text x='15' y='96' fill='#c9d1d9' font-size='11'>momentum direction.</text>
    <text x='15' y='130' fill='#d29922' font-size='10'>⚠️ Uses SAME speed</text>
    <text x='15' y='145' fill='#d29922' font-size='10'>for all parameters.</text>
  </g>
  <!-- Card 3: RMSprop -->
  <g transform='translate(410, 20)'>
    <rect width='180' height='180' rx='8' fill='#161b22' stroke='#30363d' stroke-width='1.5'/>
    <text x='15' y='28' fill='#bc8cff' font-size='13' font-weight='bold'>3. RMSprop</text>
    <text x='15' y='46' fill='#8b949e' font-size='10'>Adaptive Step Size</text>
    <text x='15' y='80' fill='#c9d1d9' font-size='11'>Gives each parameter</text>
    <text x='15' y='96' fill='#c9d1d9' font-size='11'>its own step size.</text>
    <text x='15' y='130' fill='#bc8cff' font-size='10'>⚠️ Lacks momentum</text>
    <text x='15' y='145' fill='#bc8cff' font-size='10'>stabilization.</text>
  </g>
  <!-- Card 4: AdamW -->
  <g transform='translate(605, 20)'>
    <rect width='195' height='180' rx='8' fill='#161b22' stroke='#58a6ff' stroke-width='2'/>
    <text x='15' y='28' fill='#58a6ff' font-size='13' font-weight='bold'>4. AdamW (Standard!)</text>
    <text x='15' y='46' fill='#8b949e' font-size='10'>The All-Terrain Vehicle</text>
    <text x='15' y='75' fill='#3fb950' font-size='11'>✓ Momentum (m_t)</text>
    <text x='15' y='93' fill='#3fb950' font-size='11'>✓ Adaptive Scale (v_t)</text>
    <text x='15' y='111' fill='#3fb950' font-size='11'>✓ Decoupled Decay</text>
    <text x='15' y='140' fill='#58a6ff' font-size='10' font-weight='bold'>👑 King of Foundation LLMs</text>
  </g>
</svg>
</div>

---
## 2. Why Do LLMs Break Simple Optimizers (SGD)?

In language and code, words follow <b>Zipf's Law</b>:
<ul>
  <li>Common tokens (spaces, <code>def</code>, <code>import</code>) appear millions of times. Their gradients are <b>massive (e.g. 100.0)</b>.</li>
  <li>Rare tokens (specialized variables, rare syntax) appear once in 10,000 steps. Their gradients are <b>tiny (e.g. 0.001)</b>.</li>
</ul>

<b>The Dilemma for SGD:</b><br>
- If learning rate is large enough for rare words to learn &rarr; Common words explode into NaNs and crash!<br>
- If learning rate is small enough to keep common words safe &rarr; Rare words learn nothing!

---
## 3. Exercise 1: How AdamW Auto-Scales Gradients on Paper 📝

AdamW tracks two running averages for every parameter:
<ol>
  <li><b>m_t (Momentum):</b> Direction of recent gradients.</li>
  <li><b>v_t (Variance):</b> Magnitude squared of recent gradients.</li>
</ol>

The step size for any parameter is approximately:
<pre style='font-size: 15px; background: #161b22; color: #58a6ff; padding: 12px; border-radius: 8px;'>
Effective Step ≈ m_t / sqrt(v_t)
</pre>

### Let's Test Two Extreme Parameters on Paper:
1. <b>Common Word Parameter (Huge Gradient = 100.0):</b><br>
   m_t ≈ 100.0, &nbsp; v_t ≈ 100.0² = 10,000<br>
   Step = 100.0 / sqrt(10,000) = 100.0 / 100.0 = <b>1.0</b> &nbsp; (Tamed down safely!)
<br><br>
2. <b>Rare Word Parameter (Tiny Gradient = 0.001):</b><br>
   m_t ≈ 0.001, &nbsp; v_t ≈ 0.001² = 0.000001<br>
   Step = 0.001 / sqrt(0.000001) = 0.001 / 0.001 = <b>1.0</b> &nbsp; (Boosted so it actually learns!)

<br>
<b>The Magic of AdamW:</b> Every parameter in the 100-billion-parameter model progresses at a normalized, stable speed!

---
## 4. The Parameter Splitting Rule: 2D Matrices vs. 1D RMSNorm Scales

<div style='display: flex; justify-content: center; margin: 20px 0;'>
<svg width='750' height='220' viewBox='0 0 750 220' xmlns='http://www.w3.org/2000/svg' style='background: #0d1117; border: 1px solid #30363d; border-radius: 12px; font-family: -apple-system, sans-serif;'>
  <!-- Root Node -->
  <rect x='250' y='18' width='250' height='42' rx='6' fill='#1f2937' stroke='#58a6ff' stroke-width='1.5'/>
  <text x='375' y='44' fill='#58a6ff' font-size='13' font-weight='bold' text-anchor='middle'>All Model Parameters</text>
  
  <!-- Fork Lines -->
  <path d='M 320 60 L 320 95 L 180 95 L 180 120' fill='none' stroke='#484f58' stroke-width='2'/>
  <path d='M 430 60 L 430 95 L 570 95 L 570 120' fill='none' stroke='#484f58' stroke-width='2'/>
  
  <!-- Left Card: Group 1 -->
  <rect x='60' y='120' width='240' height='80' rx='8' fill='#161b22' stroke='#3fb950' stroke-width='1.5'/>
  <text x='75' y='142' fill='#3fb950' font-size='12' font-weight='bold'>Group 1: 2D Weight Matrices</text>
  <text x='75' y='160' fill='#8b949e' font-size='10'>w_gate, w_up, w_down, q_proj, k_proj</text>
  <rect x='75' y='168' width='120' height='20' rx='4' fill='#238636'/>
  <text x='135' y='182' fill='#ffffff' font-size='10' font-weight='bold' text-anchor='middle'>Weight Decay = 0.1 (ON)</text>
  
  <!-- Right Card: Group 2 -->
  <rect x='450' y='120' width='240' height='80' rx='8' fill='#161b22' stroke='#f85149' stroke-width='1.5'/>
  <text x='465' y='142' fill='#f85149' font-size='12' font-weight='bold'>Group 2: 1D Parameters</text>
  <text x='465' y='160' fill='#8b949e' font-size='10'>RMSNorm scales (gamma), Biases</text>
  <rect x='465' y='168' width='130' height='20' rx='4' fill='#da3633'/>
  <text x='530' y='182' fill='#ffffff' font-size='10' font-weight='bold' text-anchor='middle'>Weight Decay = 0.0 (OFF!)</text>
</svg>
</div>

Why must 1D RMSNorm scales have <code>weight_decay = 0.0</code>?<br>
Because if weight decay shrinks the RMSNorm scale gamma to 0.0, the entire layer goes silent and the model dies!

---
## 5. Verify Parameter Splitting with Real Project Optimizer Code

Now let's run our actual <code>configure_optimizers</code> function from <code>src/engine/optim.py</code> on our Transformer model and verify the parameter groups:

In [ ]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.model.transformer import TransformerDecoder, ModelConfig
from src.engine.optim import configure_optimizers

# Instantiate micro model
config = ModelConfig(vocab_size=1000, dim=128, n_layers=2, n_heads=4, n_kv_heads=2)
model = TransformerDecoder(config)

# Configure optimizer with 0.1 weight decay
optimizer = configure_optimizers(model, weight_decay=0.1, learning_rate=1e-3, betas=(0.9, 0.95))

print(f"\nOptimizer Groups Created: {len(optimizer.param_groups)}")
for i, group in enumerate(optimizer.param_groups):
    sample_param = group['params'][0]
    print(f"Group {i+1}: Weight Decay = {group['weight_decay']:<4} | Parameter Count = {len(group['params']):<3} | Sample Shape = {list(sample_param.shape)}")

# Verify that Group 2 has strictly weight_decay == 0.0
assert optimizer.param_groups[1]['weight_decay'] == 0.0
print("\nSUCCESS: RMSNorm and 1D parameters are 100% protected with weight_decay = 0.0!")